# Checkpoint 4 — Recent temperature summaries and trend

**Goal:** describe several eligible readings together. Run independently with `.venv`; no previous notebook needs to run.

Latest temperature gives one observation. A recent **average** describes the level across observations, **maximum** records the highest observation, and **trend** describes warming or cooling over time. These are candidate features, not proven predictors.

A **lookback window** specifies which past measurement times we summarize. This is separate from the **six-hour prediction horizon**, which specifies the future outcome. A three-hour lookback does not change our six-hour target.

For this experiment use the half-open history window **(decision time − window, decision time]**. A measurement exactly at the left boundary is excluded; one exactly at the decision time is included only if received by then. Three hours is an illustrative starting choice, not an optimized or final choice.


In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the candidate repository.")

def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("An explicit timezone is required.")
    return value.astimezone(timezone.utc)

with (ROOT / "data/events.jsonl").open() as handle:
    events = [json.loads(line) for line in handle if line.strip()]
versions = {}
for event in events:
    versions.setdefault(event["event_id"], set()).add(event["revision"])
print("Loaded", len(events), "deliveries; no previous notebook is required.")


def known_revisions(records, checkpoint):
    """Select the highest revision received by the checkpoint for each event ID."""
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint must include a timezone")
    checkpoint = checkpoint.astimezone(timezone.utc)
    delivered = {}
    latest = {}
    for event in records:
        if utc(event["received_at"]) > checkpoint:
            continue
        key = (event["event_id"], event["revision"])
        # Compare canonical timestamps so equivalent timezone notation agrees.
        normalized = dict(event)
        for field in ("device_time", "received_at"):
            normalized[field] = utc(event[field]).isoformat()
        signature = json.dumps(normalized, sort_keys=True, allow_nan=False)
        if key in delivered:
            if delivered[key] != signature:
                raise ValueError("Conflicting content for the same event/revision")
            continue
        delivered[key] = signature
        prior = latest.get(event["event_id"])
        if prior is not None and prior["shipment_id"] != event["shipment_id"]:
            raise ValueError("An event ID changed shipment")
        if prior is None or event["revision"] > prior["revision"]:
            latest[event["event_id"]] = dict(event)
    return [latest[event_id] for event_id in sorted(latest)]


Loaded 11019 deliveries; no previous notebook is required.


## 1. Work through the arithmetic first

Suppose all three readings are available at 11 AM:

| Measurement time | Temperature |
|---|---:|
| 9 AM | 4°C |
| 10 AM | 5°C |
| 11 AM | 8°C |

With a three-hour history window, all three qualify:
- **Count:** 3 distinct selected events.
- **Average:** (4 + 5 + 8) / 3 = 5.67°C.
- **Maximum:** 8°C.
- **Trend:** fit a straight line through temperature versus elapsed hours; slope is +2°C per hour in this example.

A positive slope means warming; a negative slope means cooling. This describes past readings, not a claim that the temperature will keep changing at that rate.

For irregular timestamps, use elapsed hours rather than row numbers. Our least-squares slope uses all observations. An endpoint slope is simpler but ignores the middle readings; a more robust slope could resist outliers but adds complexity. With fewer than two distinct measurement times, trend is missing rather than invented as zero.

The average here weights each observation equally. It is not a time-weighted average: bursts of distinct readings can dominate it. Duplicate deliveries are removed, but distinct events with identical measurement times still count separately. We make this limitation explicit.


In [2]:
import math
from statistics import mean

def window_features(records, shipment, checkpoint, window_hours=3):
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint needs an explicit timezone")
    if not math.isfinite(window_hours) or window_hours <= 0:
        raise ValueError("Window must be finite and positive")
    left = checkpoint - timedelta(hours=window_hours)
    points = []
    for event in known_revisions(records, checkpoint):
        if event["shipment_id"] != shipment or event["kind"] != "temperature_c":
            continue
        value = event["value"]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            continue
        measured, received = utc(event["device_time"]), utc(event["received_at"])
        # Use the same clock rule as the latest-temperature feature.
        if measured > received or not left < measured <= checkpoint:
            continue
        points.append((measured, event["event_id"], float(value)))
    points.sort(key=lambda p: (p[0], p[1]))
    if not points:
        return {"temperature_count": 0, "temperature_mean_c": None,
                "temperature_max_c": None, "temperature_trend_c_per_hour": None,
                "temperature_span_hours": None}
    values = [p[2] for p in points]
    hours = [(p[0] - points[0][0]).total_seconds() / 3600 for p in points]
    x_mean, y_mean = mean(hours), mean(values)
    denominator = sum((x - x_mean)**2 for x in hours)
    slope = (sum((x - x_mean)*(y - y_mean) for x, y in zip(hours, values))
             / denominator) if denominator > 0 else None
    return {"temperature_count": len(points), "temperature_mean_c": y_mean,
            "temperature_max_c": max(values), "temperature_trend_c_per_hour": slope,
            "temperature_span_hours": hours[-1]}

# Invented teaching data, not a domain safety threshold.
def teaching_event(hour, value):
    timestamp = f"2026-01-01T{hour:02}:00:00Z"
    return {"event_id": f"example-{hour}", "revision": 1,
            "shipment_id": "example-shipment", "device_time": timestamp,
            "received_at": timestamp, "kind": "temperature_c", "value": value,
            "source": "teaching-sensor", "payload": {}}
readings = [teaching_event(9, 4.0), teaching_event(10, 5.0), teaching_event(11, 8.0)]
checkpoint = utc("2026-01-01T11:00:00Z")
for width in (1, 2, 3):
    print(f"{width}-hour window:", window_features(readings, "example-shipment", checkpoint, width))


1-hour window: {'temperature_count': 1, 'temperature_mean_c': 8.0, 'temperature_max_c': 8.0, 'temperature_trend_c_per_hour': None, 'temperature_span_hours': 0.0}
2-hour window: {'temperature_count': 2, 'temperature_mean_c': 6.5, 'temperature_max_c': 8.0, 'temperature_trend_c_per_hour': 3.0, 'temperature_span_hours': 1.0}
3-hour window: {'temperature_count': 3, 'temperature_mean_c': 5.666666666666667, 'temperature_max_c': 8.0, 'temperature_trend_c_per_hour': 2.0, 'temperature_span_hours': 2.0}


## 2. Interpret the window comparison

- **One hour:** (10 AM, 11 AM] includes only the 11 AM reading. Average and maximum are 8°C; trend is missing.
- **Two hours:** (9 AM, 11 AM] includes 10 AM and 11 AM. Average is 6.5°C and trend is +3°C/hour.
- **Three hours:** (8 AM, 11 AM] includes all three. Average is about 5.67°C and trend is +2°C/hour.

Short windows emphasize recent changes but can be sparse or noisy. Longer windows supply more context but may dilute a recent change. Later we can compare a small number of windows using validation data, without tuning on the final test set. These comparisons are descriptive, not evidence that one window predicts incidents better.

We also return the actual observation span. Three readings spanning ten minutes provide different evidence than three spanning two hours, even if both lie in the same three-hour window.


In [3]:
three = window_features(readings, "example-shipment", checkpoint, 3)
assert three["temperature_count"] == 3
assert math.isclose(three["temperature_mean_c"], 17/3)
assert three["temperature_max_c"] == 8
assert math.isclose(three["temperature_trend_c_per_hour"], 2)
assert window_features(readings * 2, "example-shipment", checkpoint, 3) == three
assert window_features(list(reversed(readings)), "example-shipment", checkpoint, 3) == three
assert window_features(readings, "example-shipment", checkpoint, 2)["temperature_count"] == 2
assert window_features(readings, "example-shipment", checkpoint, 1)["temperature_trend_c_per_hour"] is None
assert window_features([], "example-shipment", checkpoint, 3)["temperature_mean_c"] is None
late = {**teaching_event(10, 99.0), "event_id": "late", "received_at": "2026-01-01T12:00:00Z"}
assert window_features(readings + [late], "example-shipment", checkpoint, 3) == three
correction = {**readings[0], "revision": 2, "value": 50.0, "received_at": "2026-01-01T12:00:00Z"}
assert window_features(readings + [correction], "example-shipment", checkpoint, 3) == three
# Irregular times on a known straight line: +2°C/hour.
irregular = [teaching_event(9, 4.0),
             {**teaching_event(10, 5.0), "device_time": "2026-01-01T09:30:00Z"},
             teaching_event(11, 8.0)]
assert math.isclose(window_features(irregular, "example-shipment", checkpoint)["temperature_trend_c_per_hour"], 2)
flat = [{**e, "value": 4.0} for e in readings]
assert window_features(flat, "example-shipment", checkpoint)["temperature_trend_c_per_hour"] == 0
same_time = [readings[0], {**readings[0], "event_id": "another-event", "value": 5.0}]
assert window_features(same_time, "example-shipment", checkpoint)["temperature_trend_c_per_hour"] is None
print("Passed: arithmetic, duplicates, order, window boundaries, missing/flat trend, irregular timestamps, late data and future corrections.")


Passed: arithmetic, duplicates, order, window boundaries, missing/flat trend, irregular timestamps, late data and future corrections.


## 3. Inspect real data without selecting a winning window

Select the earliest shipment from the checkpoint file and inspect its last supplied checkpoint. Compare window sizes at that same moment. Notice the observed count and span, not just the average.


In [4]:
with (ROOT / "data/decision_times.jsonl").open() as handle:
    decisions = [json.loads(line) for line in handle if line.strip()]
first = min(decisions, key=lambda d: (utc(d["decision_time"]), d["shipment_id"]))
shipment = first["shipment_id"]
when = max(utc(d["decision_time"]) for d in decisions if d["shipment_id"] == shipment)
records = [e for e in events if e["shipment_id"] == shipment]
print("Shipment:", shipment, "Checkpoint:", when.isoformat())
for width in (1, 3, 6):
    print(f"{width}-hour history:", window_features(records, shipment, when, width))


Shipment: s-00000 Checkpoint: 2026-01-01T14:00:00+00:00
1-hour history: {'temperature_count': 1, 'temperature_mean_c': 9.857, 'temperature_max_c': 9.857, 'temperature_trend_c_per_hour': None, 'temperature_span_hours': 0.0}
3-hour history: {'temperature_count': 3, 'temperature_mean_c': 8.977666666666666, 'temperature_max_c': 9.857, 'temperature_trend_c_per_hour': 1.1124999999999998, 'temperature_span_hours': 2.0}
6-hour history: {'temperature_count': 6, 'temperature_mean_c': 7.152, 'temperature_max_c': 9.857, 'temperature_trend_c_per_hour': 1.138, 'temperature_span_hours': 5.0}


## 4. Decision record and discussion

**Provisional choices:** observation-weighted mean, maximum, least-squares temperature slope in °C/hour, explicit window boundaries, count and actual time span. We reuse revision selection before aggregation and the previous strict clock policy. No model, feature shortlist, final lookback, or incident label policy has been chosen.

**Limitations:** sensor bursts can bias the mean and slope; extrema and slopes can be sensitive to bad measurements; plausible but wrong old clocks remain undetectable here. Maximum and latest may be redundant. Additional features are candidates, not an automatic improvement. When training a model, missing values need preprocessing learned only on training data.

**Interview notes:** “I summarized only available, revision-resolved readings in explicit historical windows. I used actual elapsed hours for trend, preserved missing slopes when evidence was insufficient, and recorded count and span to expose sparse observations.”

**Try explaining this:** for the teaching readings at 9, 10, and 11 AM, why does the two-hour window at 11 AM exclude the 9 AM reading? And why is a missing trend different from a trend of zero?

**Next:** once these definitions are understood, examine label eligibility and evaluation cutoffs before building a training table or choosing a model. See [the journal](../MY_LEARNINGS.md).


## Clarification — window boundaries and missing trend

`(start, end]` excludes the lower boundary and includes the upper boundary. This is our historical-window convention, not an ML requirement. At 11 AM a two-hour window `(9 AM, 11 AM]` excludes exactly 9 AM and includes exactly 11 AM if already received.

Why choose it? It includes the current observation and assigns a shared boundary to just one interval when considering adjacent, non-overlapping intervals. Rolling windows can still overlap and reuse readings. We could use `[start, end]` for historical features instead, with consistent implementation and tests. The README separately mandates `(as_of, as_of + 6 hours]` for future labels; that is a fixed requirement, not this optional feature convention.

Missing trend means insufficient observations at distinct measurement times. Zero trend means an estimated slope of zero from usable observations. Two records at the same timestamp do not suffice. A fitted zero slope can also arise when increases and decreases balance; it does not prove every reading was constant.
